# Part I — Exploratory Data Visualization

This notebook performs the exploratory portion of the Prosper Loan Dataset project. It includes dataset loading, cleaning, overview inspection, and seven visualizations that examine loan amount, borrower rate, credit score, income, and loan status relationships.

The exploratory section is organized to answer specific questions and document observations for each visualization.

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

sns.set_theme(style='whitegrid')

plt.rcParams['figure.figsize'] = (10, 6)


## 1. Load and clean the dataset

The first step is to load the Prosper Loan dataset, inspect the columns, and apply preliminary cleaning to support the analysis.

In [ ]:
# Load the raw data and inspect the shape
loan_df = pd.read_csv('prosperLoanData.csv')
print('Original dataset shape:', loan_df.shape)
loan_df.head()

In [ ]:
# Inspect the raw dataset structure and missing values
print(loan_df.info())
print('
Missing values by column:')
print(loan_df[['LoanOriginalAmount', 'BorrowerRate', 'LoanStatus', 'IncomeRange', 'CreditScoreRangeUpper', 'DebtToIncomeRatio']].isna().sum())

# Dataset overview and preliminary wrangling

The dataset has been reduced to the variables required for the project. Missing or invalid income values were removed, borrower rates were converted to percentages, and income ranges were ordered for meaningful comparisons.


In [ ]:
# Load the Prosper Loan dataset
loan_df = pd.read_csv('prosperLoanData.csv')

# Select the variables required for the project
selected_columns = [
    'LoanOriginalAmount',
    'BorrowerRate',
    'LoanStatus',
    'IncomeRange',
    'EmploymentStatus',
    'CreditScoreRangeUpper',
    'DebtToIncomeRatio'
]
loan_df = loan_df[selected_columns].copy()

# Convert borrower rate into a percentage for easier interpretation
loan_df['BorrowerRatePercent'] = loan_df['BorrowerRate'] * 100

# Define an ordered income category and clean invalid values
income_order = [
    '$1-24,999',
    '$25,000-49,999',
    '$50,000-74,999',
    '$75,000-99,999',
    '$100,000+'
]
loan_df['IncomeRange'] = pd.Categorical(
    loan_df['IncomeRange'],
    categories=income_order,
    ordered=True
)

loan_df = loan_df.dropna(subset=[
    'LoanOriginalAmount',
    'BorrowerRate',
    'LoanStatus',
    'IncomeRange',
    'CreditScoreRangeUpper',
    'DebtToIncomeRatio'
])
loan_df = loan_df[~loan_df['IncomeRange'].isin(['Not displayed', 'Not employed', '$0'])]

loan_df.reset_index(drop=True, inplace=True)


In [ ]:
# Review cleaned dataset summary statistics
print('Cleaned dataset shape:', loan_df.shape)
print('
Income range distribution:')
print(loan_df['IncomeRange'].value_counts(dropna=False))
print('
Loan status distribution:')
print(loan_df['LoanStatus'].value_counts(dropna=False))

## Visualization 1: Histogram of Loan Amounts

**Question:** What is the distribution of loan amounts in the Prosper dataset?

**Visualization:** A histogram shows how loan sizes are distributed across the dataset.

**Observation:** The distribution is right-skewed, with most loans issued at lower amounts and a long tail of larger loans.

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(loan_df['LoanOriginalAmount'], bins=30, color='#4c72b0', edgecolor='white')
plt.title('Distribution of Loan Original Amounts', fontsize=14, weight='bold')
plt.xlabel('Loan Original Amount ($)')
plt.ylabel('Number of Loans')
plt.tight_layout()
plt.show()

## Visualization 2: Count Plot of Loan Status

**Question:** Which loan statuses occur most frequently in the dataset?

**Visualization:** A count plot shows the frequency of each loan status category.

**Observation:** Active and completed loans dominate the dataset, while other statuses appear much less often.

In [ ]:
plt.figure(figsize=(10, 6))
status_order = loan_df['LoanStatus'].value_counts().index
sns.countplot(data=loan_df, y='LoanStatus', order=status_order, palette='crest')
plt.title('Loan Status Counts', fontsize=14, weight='bold')
plt.xlabel('Number of Loans')
plt.ylabel('Loan Status')
plt.tight_layout()
plt.show()

## Visualization 3: Credit Score vs Borrower Rate Scatter Plot

**Question:** Is there a relationship between borrower credit score and interest rate?

**Visualization:** A scatter plot compares credit score range upper values to borrower interest rates.

**Observation:** Higher credit scores are associated with lower borrower interest rates, indicating a negative relationship.

In [ ]:
plt.figure(figsize=(10, 6))
sample_df = loan_df.sample(n=5000, random_state=42)
sns.scatterplot(
    data=sample_df,
    x='CreditScoreRangeUpper',
    y='BorrowerRatePercent',
    alpha=0.4,
    color='#ff7f0e'
)
plt.title('Credit Score vs Borrower Interest Rate', fontsize=14, weight='bold')
plt.xlabel('Credit Score Range Upper')
plt.ylabel('Borrower Rate (%)')
plt.tight_layout()
plt.show()

## Visualization 4: Income Range vs Loan Amount Box Plot

**Question:** How does borrower income range relate to the size of loans received?

**Visualization:** A box plot compares loan amounts across ordered income ranges.

**Observation:** Higher income ranges tend to borrow larger loan amounts, and the median loan size increases with income.

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(
    data=loan_df,
    x='IncomeRange',
    y='LoanOriginalAmount',
    order=loan_df['IncomeRange'].cat.categories,
    palette='Spectral'
)
plt.title('Loan Amount by Income Range', fontsize=14, weight='bold')
plt.xlabel('Income Range')
plt.ylabel('Loan Original Amount ($)')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## Visualization 5: Correlation Heatmap for Numerical Variables

**Question:** What relationships exist among the numerical loan variables?

**Visualization:** A heatmap shows correlations between loan amount, borrower rate, credit score, and debt-to-income ratio.

**Observation:** Borrower rate has a negative correlation with credit score, and loan size has moderate relationships with other financial metrics.

In [ ]:
numeric_features = [
    'LoanOriginalAmount',
    'BorrowerRatePercent',
    'CreditScoreRangeUpper',
    'DebtToIncomeRatio'
]
corr_matrix = loan_df[numeric_features].corr()
plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='Blues', linewidths=0.5)
plt.title('Correlation Heatmap of Key Numerical Variables', fontsize=14, weight='bold')
plt.tight_layout()
plt.show()

## Visualization 6: Facet Plot by Income Range

**Question:** Does the credit score versus borrower rate pattern vary across income groups?

**Visualization:** A faceted scatter plot shows the relationship for each income range.

**Observation:** The negative relationship between credit score and borrower rate is present across all income groups, although higher-income borrowers are more concentrated in lower-rate regions.

In [ ]:
facet_sample = loan_df.sample(n=8000, random_state=42)
g = sns.FacetGrid(
    facet_sample,
    col='IncomeRange',
    col_wrap=3,
    height=4,
    sharey=True
)
g.map_dataframe(
    sns.scatterplot,
    x='CreditScoreRangeUpper',
    y='BorrowerRatePercent',
    alpha=0.35,
    color='#2ca02c'
)
g.set_axis_labels('Credit Score Range Upper', 'Borrower Rate (%)')
g.fig.suptitle('Borrower Rate vs Credit Score Across Income Groups', fontsize=16, weight='bold', y=1.02)
plt.tight_layout()
plt.show()

## Visualization 7: Multivariate Scatter Plot

**Question:** How do loan amount, borrower rate, loan status, and credit score interact?

**Visualization:** A multivariate scatter plot uses color and size to display multiple dimensions simultaneously.

**Observation:** Larger credit scores are generally concentrated at lower borrower rates, and loan status categories overlap but show distinct clusters in loan amount and rate space.

In [ ]:
multi_sample = loan_df.sample(n=5000, random_state=42)
plt.figure(figsize=(11, 7))
sns.scatterplot(
    data=multi_sample,
    x='LoanOriginalAmount',
    y='BorrowerRatePercent',
    hue='LoanStatus',
    size='CreditScoreRangeUpper',
    sizes=(20, 200),
    alpha=0.45,
    palette='tab10'
)
plt.title('Loan Amount, Borrower Rate, Loan Status, and Credit Score', fontsize=14, weight='bold')
plt.xlabel('Loan Original Amount ($)')
plt.ylabel('Borrower Rate (%)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0)
plt.tight_layout()
plt.show()

## Exploratory Summary

This exploratory section demonstrated the distribution of loan amounts, the importance of loan status, and the relationships between borrower rate, credit score, income range, and debt-to-income ratio. These visualizations establish the foundation for a polished explanatory story in Part II.